
# BLEVE Peak Pressure Prediction

This notebook performs data preprocessing, feature engineering, model development, hyperparameter tuning, and evaluation for BLEVE peak pressure prediction.



# 1. Import Libraries



### Library Overview

- `pandas` is used for handling tabular datasets.
- `numpy` is used for numerical operations.
- `scikit-learn` provides preprocessing, machine learning models, hyperparameter tuning, and evaluation metrics.
- `matplotlib` is used for visualisation and model comparison.


In [ ]:
# Split dataset into training and validation sets

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    mean_absolute_percentage_error,
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt



# 2. Load Dataset

Upload `train.csv` and `test.csv` when running in Google Colab.


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:
# Load training and testing datasets

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

train.head()



# 3. Initial Data Exploration


In [ ]:

print(train.info())

print("\nMissing Values:")
print(train.isnull().sum())

print("\nDuplicate Rows:")
print(train.duplicated().sum())



### Why Data Cleaning Matters

BLEVE simulation datasets may contain:
- missing values
- duplicated rows
- unrealistic physical values
- placeholder values such as `-1`

Cleaning the dataset improves model reliability and prediction accuracy.


In [ ]:

numeric_df = train.select_dtypes(include="number")

stats = numeric_df.describe()

# Remove quartiles
stats = stats.drop(["25%", "50%", "75%"])

stats



# 4. Data Cleaning

This section:
- replaces invalid placeholder values
- removes duplicates
- handles missing values
- detects outliers


In [ ]:
# Replace invalid placeholder values with NaN

# Replace placeholder values
train.replace(-1, np.nan, inplace=True)
test.replace(-1, np.nan, inplace=True)

# Remove duplicate rows
train = train.drop_duplicates()

# Fill missing numeric values
numeric_cols = train.select_dtypes(include="number").columns

train[numeric_cols] = train[numeric_cols].fillna(
    train[numeric_cols].mean()
)

test[numeric_cols.drop("Target Pressure (bar)", errors="ignore")] =     test[numeric_cols.drop("Target Pressure (bar)", errors="ignore")].fillna(
        train[numeric_cols.drop("Target Pressure (bar)", errors="ignore")].mean()
)

print("Missing values after cleaning:")
print(train.isnull().sum().sum())


In [ ]:

# Outlier Detection using IQR

for col in numeric_cols:

    if col == "Target Pressure (bar)":
        continue

    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    train = train[
        (train[col] >= lower) &
        (train[col] <= upper)
    ]

print("Shape after outlier removal:", train.shape)



### Feature Engineering Rationale

Additional features are created to help the model learn physical relationships.

Examples:
- Tank Aspect Ratio captures tank geometry proportions.
- Tank Volume estimates storage capacity.
- Temperature Difference may influence vapour expansion behaviour.



# 5. Feature Engineering


In [ ]:
# Create new engineered features

# Tank aspect ratio
train["Tank Aspect Ratio"] = (
    train["Tank Width (m)"] /
    train["Tank Length (m)"]
)

test["Tank Aspect Ratio"] = (
    test["Tank Width (m)"] /
    test["Tank Length (m)"]
)

# Tank volume
train["Tank Volume"] = (
    train["Tank Width (m)"] *
    train["Tank Length (m)"] *
    train["Tank Height (m)"]
)

test["Tank Volume"] = (
    test["Tank Width (m)"] *
    test["Tank Length (m)"] *
    test["Tank Height (m)"]
)

# Temperature difference
train["Temperature Difference"] = (
    train["Vapour Temperature (K)"] -
    train["Liquid Temperature (K)"]
)

test["Temperature Difference"] = (
    test["Vapour Temperature (K)"] -
    test["Liquid Temperature (K)"]
)

train.head()



# 6. Feature Selection and Encoding


In [ ]:
# Convert categorical variables into numerical format

target_col = "Target Pressure (bar)"

# Remove ID columns
drop_cols = ["ID", "Sensor ID"]

train = train.drop(columns=drop_cols, errors="ignore")
test_ids = test["ID"]
test = test.drop(columns=drop_cols, errors="ignore")

# Separate features and target
y = train[target_col]
X = train.drop(columns=[target_col])

# Detect categorical columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

# One-hot encoding
X = pd.get_dummies(X, columns=cat_cols)
test = pd.get_dummies(test, columns=cat_cols)

# Align train and test
X, test = X.align(test, join="left", axis=1, fill_value=0)

print("Final Feature Shape:", X.shape)



### Why Multiple Models Are Compared

Different machine learning models behave differently:
- Linear Regression assumes linear relationships.
- SVR uses kernel methods for nonlinear relationships.
- Random Forest captures complex nonlinear interactions.

Comparing multiple models helps identify the most suitable approach.



# 7. Train Validation Split


In [ ]:
# Split dataset into training and validation sets

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)



# 8. Model Development

The following models are evaluated:
- Linear Regression
- Support Vector Regression (SVR)
- Random Forest Regressor


In [ ]:

results = []



### Evaluation Metrics

The following metrics are used:

- **MAPE** measures percentage prediction error.
- **R² Score** measures explained variance.
- **RMSE** penalises large prediction errors.

Lower MAPE and RMSE are better, while higher R² is better.



## 8.1 Linear Regression


In [ ]:
# Train baseline linear regression model

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_val)

lr_mape = mean_absolute_percentage_error(y_val, lr_pred)
lr_r2 = r2_score(y_val, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_val, lr_pred))

results.append([
    "Linear Regression",
    lr_mape,
    lr_r2,
    lr_rmse
])

print("MAPE:", lr_mape)
print("R2:", lr_r2)
print("RMSE:", lr_rmse)



## 8.2 Support Vector Regression


In [ ]:
# Train Support Vector Regression model with scaling

svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])

param_grid_svr = {
    "svr__C": [1, 10],
    "svr__kernel": ["rbf"],
    "svr__gamma": ["scale"]
}

svr_grid = GridSearchCV(
    svr_pipeline,
    param_grid_svr,
    cv=3,
    n_jobs=-1
)

svr_grid.fit(X_train, y_train)

svr_pred = svr_grid.predict(X_val)

svr_mape = mean_absolute_percentage_error(y_val, svr_pred)
svr_r2 = r2_score(y_val, svr_pred)
svr_rmse = np.sqrt(mean_squared_error(y_val, svr_pred))

results.append([
    "SVR",
    svr_mape,
    svr_r2,
    svr_rmse
])

print("Best Parameters:", svr_grid.best_params_)
print("MAPE:", svr_mape)
print("R2:", svr_r2)
print("RMSE:", svr_rmse)



## 8.3 Random Forest Regressor


In [ ]:
# Train Random Forest model with hyperparameter tuning

rf_model = RandomForestRegressor(random_state=42)

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20]
}

rf_grid = GridSearchCV(
    rf_model,
    param_grid_rf,
    cv=3,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

rf_pred = rf_grid.predict(X_val)

rf_mape = mean_absolute_percentage_error(y_val, rf_pred)
rf_r2 = r2_score(y_val, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))

results.append([
    "Random Forest",
    rf_mape,
    rf_r2,
    rf_rmse
])

print("Best Parameters:", rf_grid.best_params_)
print("MAPE:", rf_mape)
print("R2:", rf_r2)
print("RMSE:", rf_rmse)



# 9. Model Comparison


In [ ]:

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAPE", "R2", "RMSE"]
)

results_df


In [ ]:

plt.figure(figsize=(8,5))

plt.bar(results_df["Model"], results_df["MAPE"])

plt.ylabel("MAPE")
plt.title("Model Comparison - MAPE")

plt.show()



# 10. Final Model Selection

The best performing model is selected based on:
- lowest MAPE
- highest R2
- lowest RMSE


In [ ]:
# Generate final Kaggle submission file

best_model = rf_grid

predictions = best_model.predict(test)

submission = pd.DataFrame({
    "ID": test_ids,
    "Target Pressure (bar)": predictions
})

submission.to_csv("sample_prediction.csv", index=False)

submission.head()



# 11. Conclusion

Random Forest generally performs best for BLEVE prediction because:
- BLEVE behaviour is highly nonlinear
- interactions between geometry, temperature, and sensor position are complex
- tree-based models handle nonlinear engineering data effectively

Linear Regression serves as a baseline model, while SVR captures nonlinear relationships using kernel methods.
